### Cancer Study Cleaning Steps

In [13]:
import numpy as np
import pandas as pd

In [14]:

dataset = pd.read_csv('cancer_incidents_extract.csv')

In [15]:
# Drop columns: 'Source.Name', 'Domain' and 12 other columns
dataset = dataset.drop(columns=['Source.Name', 'Domain', 'Indicator', 'Year', 'GeogID', 'Race_Ethnicity', 'Gender', 'Age_Group', 'Month', 'Measure', 'ts', 'measureName', 'contentAreaName', 'Race_Ethnicitylabel'])

In [16]:
 # Filter rows based on column: 'Name'
dataset = dataset[dataset['Name'] != "ARIZONA"]

In [17]:
# Create Gender Column to account for breast and testicular cancer 
conditions = [
     dataset['indicatorName'].str.contains('Females Only', case=False, na=False),
     dataset['indicatorName'].str.contains('Males Only', case=False, na=False),
     dataset['Genderlabel'] == 'Female',
     dataset['Genderlabel'] == 'Male'
            ]
choices = ['Female', 'Male', 'Female', 'Male']
dataset['gender'] = np.select(conditions, choices, default=np.nan)

In [18]:
# drop 'Genderlabel'
dataset = dataset.drop(columns=['Genderlabel'])

In [19]:
# Rename columns
dataset = dataset.rename(columns={'Name': 'county', 'Value': 'cancer_rate', 'indicatorName': 'cancer_type'})

In [20]:

# create primary key

# extract cancer name and gender code
dataset['cancer_name'] = dataset['cancer_type'].str.split(' ').str[2]
dataset['gender_code'] = dataset['gender'].str[0]
# create unique composite key of county, cancer_name, and gender_code
dataset['ID'] = (dataset['county'] + '-' + dataset['cancer_name'] + '-' + dataset['gender_code']).str.upper()
# remove intermediate columns
dataset = dataset.drop(columns=['cancer_name', 'gender_code'])

In [21]:
# move ['ID'] to front
cols_to_move = ['ID']
remaining_cols = [col for col in dataset.columns if col not in cols_to_move]
new_order = cols_to_move + remaining_cols
dataset = dataset[new_order]

In [22]:
# extract cancer type text between delimiters 
dataset['cancer_type'] = dataset['cancer_type'].str.split(' ')
dataset['cancer_type'] = dataset['cancer_type'].str[2:].str.join(' ')

In [23]:
# Change column types to string
dataset = dataset.astype({'ID': 'string','county': 'string', 'cancer_type': 'string','gender': 'string'})

In [24]:
dataset.head()

,ID,county,cancer_rate,cancer_type,gender
1,YUMA-BLADDER-M,YUMA,24.4,Bladder Cancer,Male
2,YAVAPAI-BLADDER-M,YAVAPAI,38.4,Bladder Cancer,Male
3,SANTA CRUZ-BLADDER-M,SANTA CRUZ,24.9,Bladder Cancer,Male
4,PINAL-BLADDER-M,PINAL,30.4,Bladder Cancer,Male
5,PIMA-BLADDER-M,PIMA,31.2,Bladder Cancer,Male
